# ov.fm — Unified Foundation Model API

The `ov.fm` module provides a **unified, model-agnostic API** for working with 22 single-cell foundation models. Instead of learning each model's unique interface, you can use the same 6-step workflow for any model:

1. **Discover** — Browse available models and their capabilities
2. **Profile** — Automatically analyze your dataset
3. **Select** — Let the system recommend the best model
4. **Validate** — Check data-model compatibility before running
5. **Run** — Execute inference with a single function call
6. **Interpret** — Generate QA metrics and visualizations

**Supported models include:** scGPT, Geneformer, UCE, scFoundation, CellPLM, scBERT, GeneCompass, Nicheformer, scMulan, and 13 more.

**Cite:** Zeng, Z. et al. (2024). OmicVerse: a framework for bridging and deepening insights across bulk and single-cell sequencing. *Nature Communications*, 15(1), 5983.

In [ ]:
import omicverse as ov
import scanpy as sc
import numpy as np

ov.plot_set()

## Step 1: Discover Available Models

Use `ov.fm.list_models()` to browse all registered foundation models. You can filter by task type to find models that support your specific analysis.

In [ ]:
# List all available models
all_models = ov.fm.list_models()
print(f"Total models available: {all_models['count']}")

# Display as a table
import pandas as pd
df = pd.DataFrame(all_models['models'])
df[['name', 'status', 'tasks', 'species', 'zero_shot', 'gpu_required', 'min_vram_gb']]

In [ ]:
# Filter by task: only models that support embedding
embed_models = ov.fm.list_models(task="embed")
print(f"Models supporting embedding: {embed_models['count']}")
for m in embed_models['models']:
    print(f"  - {m['name']:15s}  species={m['species']}  zero_shot={m['zero_shot']}")

### Get detailed model information

Use `ov.fm.describe_model()` to get full specifications for any model, including input/output contracts, hardware requirements, and documentation links.

In [ ]:
# Get detailed information about scGPT
info = ov.fm.describe_model("scgpt")

print("=== Model Info ===")
print(f"Name: {info['model']['name']}")
print(f"Version: {info['model']['version']}")
print(f"Tasks: {info['model']['tasks']}")
print(f"Species: {info['model']['species']}")
print(f"Embedding dim: {info['model']['embedding_dim']}")
print(f"Differentiator: {info['model']['differentiator']}")

print("\n=== Input Contract ===")
print(f"Gene ID scheme: {info['input_contract']['gene_id_scheme']}")
print(f"Gene ID notes: {info['input_contract']['gene_id_notes']}")
print(f"Preprocessing: {info['input_contract']['preprocessing']}")

print("\n=== Output Contract ===")
print(f"Embedding key: {info['output_contract']['embedding_key']}")
print(f"Embedding dim: {info['output_contract']['embedding_dim']}")

In [ ]:
# Compare multiple models side by side
models_to_compare = ["scgpt", "geneformer", "uce", "scfoundation", "cellplm"]
comparison = []
for name in models_to_compare:
    info = ov.fm.describe_model(name)
    m = info['model']
    comparison.append({
        'Model': m['name'],
        'Embedding Dim': m['embedding_dim'],
        'Gene IDs': info['input_contract']['gene_id_scheme'],
        'Species': ', '.join(m['species']),
        'Zero-shot': m['zero_shot_embedding'],
        'GPU Required': m['hardware']['gpu_required'],
        'Min VRAM (GB)': m['hardware']['min_vram_gb'],
    })
pd.DataFrame(comparison)

## Step 2: Profile Your Data

`ov.fm.profile_data()` automatically detects your dataset's species, gene identifier scheme, modality, and checks compatibility with all registered models.

First, let's prepare a test dataset.

In [ ]:
# Load example PBMC dataset
adata = sc.datasets.pbmc3k_processed()
print(f"Dataset: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"Gene names (first 5): {adata.var_names[:5].tolist()}")

# Save to h5ad for ov.fm workflow
adata.write_h5ad("pbmc3k_processed.h5ad")

In [ ]:
# Profile the dataset
profile = ov.fm.profile_data("pbmc3k_processed.h5ad")

print("=== Data Profile ===")
print(f"Cells: {profile['n_cells']:,}")
print(f"Genes: {profile['n_genes']:,}")
print(f"Species: {profile['species']}")
print(f"Gene scheme: {profile['gene_scheme']}")
print(f"Modality: {profile['modality']}")
print(f"Has raw counts: {profile['has_raw']}")
print(f"Layers: {profile['layers']}")
print(f"Batch columns: {profile['batch_columns']}")
print(f"Cell type columns: {profile['celltype_columns']}")

In [ ]:
# Check compatibility with specific models
compatible_models = []
for name, compat in profile['model_compatibility'].items():
    if compat['compatible']:
        compatible_models.append(name)
    elif compat['issues']:
        print(f"  {name}: {compat['issues']}")

print(f"\nCompatible models ({len(compatible_models)}): {compatible_models}")

## Step 3: Automatic Model Selection

`ov.fm.select_model()` analyzes your data and recommends the best model based on:
- Species and gene ID compatibility
- Task support and zero-shot capability
- Hardware requirements
- Adapter implementation readiness

In [ ]:
# Auto-select the best model for embedding
selection = ov.fm.select_model(
    "pbmc3k_processed.h5ad",
    task="embed",
    prefer_zero_shot=True,
)

print("=== Recommended ===")
print(f"Model: {selection['recommended']['name']}")
print(f"Rationale: {selection['recommended']['rationale']}")

print("\n=== Fallback Options ===")
for fb in selection['fallbacks']:
    print(f"  - {fb['name']}: {fb['rationale']}")

print(f"\nPreprocessing: {selection['preprocessing_notes']}")

In [ ]:
# Select with VRAM constraint (e.g., 8 GB GPU)
selection_8gb = ov.fm.select_model(
    "pbmc3k_processed.h5ad",
    task="embed",
    max_vram_gb=8,
)
print(f"Best model for 8GB VRAM: {selection_8gb['recommended']['name']}")
print(f"Rationale: {selection_8gb['recommended']['rationale']}")

## Step 4: Validate Data-Model Compatibility

Before running inference, use `ov.fm.preprocess_validate()` to check for potential issues and get auto-fix suggestions.

In [ ]:
# Validate data compatibility with scGPT
validation = ov.fm.preprocess_validate(
    "pbmc3k_processed.h5ad",
    model_name="scgpt",
    task="embed",
)

print(f"Status: {validation['status']}")
print(f"\nDiagnostics:")
for d in validation['diagnostics']:
    print(f"  [{d['severity']}] {d['message']}")

if validation['auto_fixes']:
    print(f"\nSuggested fixes:")
    for fix in validation['auto_fixes']:
        print(f"  - {fix}")

In [ ]:
# Validate with a model that requires Ensembl IDs (Geneformer)
validation_gf = ov.fm.preprocess_validate(
    "pbmc3k_processed.h5ad",
    model_name="geneformer",
    task="embed",
)

print(f"Status: {validation_gf['status']}")
for d in validation_gf['diagnostics']:
    print(f"  [{d['severity']}] {d['message']}")
# Note: Geneformer requires Ensembl IDs; the diagnostic will flag this
# if your data uses gene symbols

## Step 5: Run Foundation Model Inference

`ov.fm.run()` is the core execution function. It:
1. Validates data-model compatibility
2. Loads the model and checkpoint
3. Runs inference
4. Writes results (embeddings/annotations) back to AnnData
5. Records provenance metadata

### 5a. Using the high-level `ov.fm.run()` API

In [ ]:
# Run scGPT embedding
result = ov.fm.run(
    task="embed",
    model_name="scgpt",
    adata_path="pbmc3k_processed.h5ad",
    output_path="pbmc3k_scgpt.h5ad",
    device="auto",        # auto-detect GPU/CPU
    batch_size=64,
)
print(result)

In [ ]:
# Load results and visualize
adata_scgpt = sc.read_h5ad("pbmc3k_scgpt.h5ad")
print(f"Embedding key: X_scGPT")
print(f"Embedding shape: {adata_scgpt.obsm['X_scGPT'].shape}")

# Compute UMAP from scGPT embeddings
sc.pp.neighbors(adata_scgpt, use_rep='X_scGPT')
sc.tl.umap(adata_scgpt)
ov.pl.embedding(adata_scgpt, basis='X_umap', color=['louvain'])

### 5b. Using the low-level `ov.llm.SCLLMManager` API

For more fine-grained control (fine-tuning, cell type annotation, custom preprocessing), you can use the model-specific `SCLLMManager` interface directly.

In [ ]:
# Low-level API: direct model access
manager = ov.llm.SCLLMManager(
    model_type="scgpt",
    model_path="path/to/scgpt/checkpoint",
)

# Get embeddings with full control
adata = sc.read_h5ad("pbmc3k_processed.h5ad")
embeddings = manager.get_embeddings(adata, batch_size=64)
adata.obsm['X_scGPT'] = embeddings

# Fine-tune on reference data
ref_adata = adata[adata.obs['louvain'].isin(['CD4 T cells', 'CD8 T cells', 'B cells'])]
ref_adata.obs['celltype'] = ref_adata.obs['louvain'].copy()
manager.fine_tune(
    train_adata=ref_adata,
    task="annotation",
    epochs=5,
)

# Predict cell types
predictions = manager.predict_celltypes(adata)

## Step 6: Interpret Results

`ov.fm.interpret_results()` generates QA metrics for model outputs, including embedding dimensionality, silhouette scores, and provenance tracking.

In [ ]:
# Interpret results from the scGPT run
interpretation = ov.fm.interpret_results(
    "pbmc3k_scgpt.h5ad",
    task="embed",
)

print("=== QA Metrics ===")
print(f"Cells: {interpretation['metrics']['n_cells']:,}")
print(f"Genes: {interpretation['metrics']['n_genes']:,}")

if 'embeddings' in interpretation['metrics']:
    for key, info in interpretation['metrics']['embeddings'].items():
        print(f"\nEmbedding '{key}':")
        print(f"  Dimensions: {info['dim']}")
        print(f"  Cells: {info['n_cells']}")
        if 'silhouette' in info:
            print(f"  Silhouette score: {info['silhouette']:.4f}")

if 'provenance' in interpretation['metrics']:
    print(f"\nProvenance: {interpretation['metrics']['provenance']}")

## Multi-Model Comparison

One of the key strengths of `ov.fm` is the ability to run multiple models on the same dataset and compare results. This section demonstrates how to benchmark models side by side.

In [ ]:
# Run multiple models on the same dataset
models_to_run = ["scgpt", "geneformer", "uce"]
results = {}

for model_name in models_to_run:
    print(f"\n{'='*50}")
    print(f"Running {model_name}...")
    result = ov.fm.run(
        task="embed",
        model_name=model_name,
        adata_path="pbmc3k_processed.h5ad",
        output_path=f"pbmc3k_{model_name}.h5ad",
        device="auto",
    )
    results[model_name] = result
    print(f"  Status: {result.get('status', result.get('error', 'unknown'))}")

In [ ]:
# Compare embeddings visually
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(models_to_run), figsize=(6*len(models_to_run), 5))

for i, model_name in enumerate(models_to_run):
    output_file = f"pbmc3k_{model_name}.h5ad"
    try:
        adata_m = sc.read_h5ad(output_file)
        # Find the embedding key for this model
        info = ov.fm.describe_model(model_name)
        emb_key = info['output_contract']['embedding_key'].split("'")[1]
        
        sc.pp.neighbors(adata_m, use_rep=emb_key)
        sc.tl.umap(adata_m)
        sc.pl.umap(adata_m, color='louvain', ax=axes[i], 
                   title=f"{model_name} ({adata_m.obsm[emb_key].shape[1]}d)",
                   show=False)
    except Exception as e:
        axes[i].set_title(f"{model_name}: {e}")

plt.tight_layout()
plt.show()

In [ ]:
# Quantitative comparison: silhouette scores
from sklearn.metrics import silhouette_score

comparison_results = []
for model_name in models_to_run:
    output_file = f"pbmc3k_{model_name}.h5ad"
    try:
        adata_m = sc.read_h5ad(output_file)
        info = ov.fm.describe_model(model_name)
        emb_key = info['output_contract']['embedding_key'].split("'")[1]
        emb = adata_m.obsm[emb_key]
        
        sil = silhouette_score(emb, adata_m.obs['louvain'])
        comparison_results.append({
            'Model': model_name,
            'Embedding Dim': emb.shape[1],
            'Silhouette Score': round(sil, 4),
        })
    except Exception as e:
        comparison_results.append({'Model': model_name, 'Error': str(e)})

pd.DataFrame(comparison_results)

## Advanced: Custom Checkpoint Paths

By default, `ov.fm` looks for checkpoints in `~/.cache/omicverse/models/<model_name>/`. You can override this via:
- The `checkpoint_dir` parameter in `ov.fm.run()`
- Environment variables: `OV_FM_CHECKPOINT_DIR_SCGPT`, `OV_FM_CHECKPOINT_DIR_GENEFORMER`, etc.
- A global base directory: `OV_FM_CHECKPOINT_DIR` with model-named subfolders

In [ ]:
# Using a custom checkpoint directory
result = ov.fm.run(
    task="embed",
    model_name="scgpt",
    adata_path="pbmc3k_processed.h5ad",
    output_path="pbmc3k_scgpt_custom.h5ad",
    checkpoint_dir="/path/to/my/scgpt/checkpoint",
)

## Advanced: Conda Subprocess Isolation

Some models have conflicting dependencies. `ov.fm` supports running models in isolated conda environments via subprocess. If a conda env named `scfm-<model_name>` exists (e.g., `scfm-scgpt`), `ov.fm.run()` will automatically use it.

```bash
# Create an isolated environment for a model
conda create -n scfm-scgpt python=3.10
conda activate scfm-scgpt
pip install omicverse scgpt
```

To disable conda subprocess execution:
```python
import os
os.environ['OV_FM_DISABLE_CONDA_SUBPROCESS'] = '1'
```

## Advanced: Plugin System

You can register custom models with `ov.fm` via the plugin system.

### Entry-point plugins (pip packages)

In your package's `pyproject.toml`:
```toml
[project.entry-points."omicverse.fm"]
my_model = "my_package.fm_plugin:register"
```

### Local plugins

Place a Python file in `~/.omicverse/plugins/fm/`:

In [ ]:
# Example: registering a custom model plugin (for illustration)
from omicverse.fm import ModelSpec, TaskType, Modality, SkillReadyStatus, OutputKeys
from omicverse.fm.adapters.base import BaseAdapter

# Define model spec
my_spec = ModelSpec(
    name="my_custom_model",
    version="v1.0",
    skill_ready=SkillReadyStatus.READY,
    tasks=[TaskType.EMBED],
    modalities=[Modality.RNA],
    species=["human"],
    output_keys=OutputKeys(embedding_key="X_my_model"),
    embedding_dim=256,
)

# Register it
registry = ov.fm.get_registry()
registry.register(my_spec, source="user")

# Now it appears in list_models
print([m['name'] for m in ov.fm.list_models()['models'] if 'custom' in m['name']])

## Model Quick Reference

| Model | Dim | Gene IDs | Species | Key Strength |
|-------|-----|----------|---------|-------------|
| **scGPT** | 512 | Symbol | human, mouse | Multi-modal (RNA+ATAC+Spatial), attention maps |
| **Geneformer** | 512 | Ensembl | human | CPU-capable, rank-value encoding, network biology |
| **UCE** | 1280 | Symbol | 7 species | Broadest species support, protein structure embeddings |
| **scFoundation** | 512 | Custom | human | Perturbation/drug response, xTrimoGene architecture |
| **CellPLM** | 512 | Symbol | human | Fastest inference, cell-centric (not gene-centric) |
| **scBERT** | 200 | Symbol | human | Lightest model, 200-dim compact embeddings |
| **Nicheformer** | 512 | Symbol | human, mouse | Spatial-aware, niche modeling |
| **scMulan** | 512 | Symbol | human | Native multi-omics (RNA+ATAC+Protein) |

For the full list of 22 models, run `ov.fm.list_models()`.

## API Reference Summary

| Function | Purpose |
|----------|--------|
| `ov.fm.list_models(task=)` | Browse available models, filter by task |
| `ov.fm.describe_model(name)` | Get full model spec and I/O contract |
| `ov.fm.profile_data(path)` | Auto-detect species, gene scheme, modality |
| `ov.fm.select_model(path, task=)` | Recommend best model for your data |
| `ov.fm.preprocess_validate(path, model, task)` | Check compatibility, get fix suggestions |
| `ov.fm.run(task=, model_name=, adata_path=)` | Execute inference |
| `ov.fm.interpret_results(path, task=)` | Generate QA metrics and visualizations |